# Lab 0-04 Walkthrough: A Bounded Forensic Agent

This notebook combines the Lab 0-04 agent concepts in one small synthetic forensic workflow. The model selects approved tools, the agent validates and runs them, working memory retains the returned observations, and the model produces a bounded final review.

## Scenario and Expected Result

A clinic supervisor submitted a clinic-issued Android phone after noticing after-hours activity involving a screenshot of a staff schedule. Your first-pass review must identify supported device activity, state what remains unknown, and recommend one next human-review step.

Do not prove a policy violation, identify who saw the screenshot, or claim successful delivery unless the available artifacts directly confirm it.

## Agent Components in This Walkthrough

| Agent concept | Forensic workflow behavior |
| --- | --- |
| Role and goal | Produce a cautious first-pass device-activity review. |
| Inputs and instructions | The user task and case-packet evidence are inputs; `agent_instructions` sets the persistent rules for using them. |
| Reasoning engine | The configured model chooses the next approved tool or a final review. |
| Environment | The synthetic case packet is the environment the agent may observe. This walkthrough does not let the agent change the phone, files, or any external system. |
| Approved tools | The model may request only the case brief, manifest, or event-log reader. |
| Tool selection and validation | The model requests one tool as JSON; the agent checks the name before executing Python. |
| Short-term memory | The agent keeps conversation context and approved evidence across turns. |
| Stop condition | Allow at most one use of each approved tool. After the tool limit, require a final review; stop only when that final review is returned. |
| Structured output and human review | The final review separates observations from unknowns and requires one next human-review step. |

## Agent Specification

Use this table as the workflow contract while reading the code. Each row describes a rule that the program or model must follow.

| Specification item | This walkthrough |
| --- | --- |
| Role | Mobile Device Activity Review Agent: a cautious first-pass reviewer of the synthetic clinic-phone case. |
| Goal | Gather needed packet information, state supported observations and unknowns, and recommend one next human-review step. |
| Inputs | The user task and the approved synthetic case packet returned by the three reader tools. |
| Instructions | `agent_instructions` is persistent control guidance: role, goal, approved-tool boundary, evidence limits, and output rules. |
| Reasoning engine | The configured `model` reads the current context and chooses a tool request or final review. |
| Approved tools | `read_case_brief`, `read_manifest`, and `read_device_events`. |
| Conversation memory (`messages`) | This short-lived memory is the full chat history: system instructions, the user task, model decisions, and current evidence context. It lets the model continue the multi-turn conversation. |
| Evidence working memory | `working_memory` is a short-lived Python list of dictionaries, each containing an approved tool name and its returned evidence. It starts empty, is sent back to the model on the next turn, and is not an external file or saved after the notebook run. |
| Model-decision format | On every turn, the model returns structured JSON with an `action` and `tool_name`. `action` is either `tool` (request one approved tool) or `final` (return the final structured review); a final decision also fills the review fields. |
| Tool-result formats | `read_case_brief` returns plain text. `read_manifest` and `read_device_events` return JSON-formatted text. The program stores every result in a working-memory dictionary. |
| Tool request and validation | The model requests one zero-argument tool in structured JSON. The program runs it only when its name is currently approved and unused. |
| Stop condition | The workflow stops when the model returns a final review. It allows at most one call to each of the three tools; when none remain, the next turn must finalize. |
| Required structured result | A final JSON review with `observations`, `unknowns`, `next_step`, and `needs_human_review`. |
| Evidence and human-review boundary | Do not claim screenshot contents, recipient viewing, or successful delivery without direct artifact support. The final review always requires human review. |

### How `action` Branches the Workflow

This is pseudocode: it describes the decision logic in plain language, not additional Python that you need to run.

```text
ask the model for a structured decision

if decision.action is `tool`:
    check that decision.tool_name is approved and unused
    run that one tool
    save its result in working_memory
    start the next turn

if decision.action is `final`:
    check that the review requires human review
    stop and show the structured review
```

## Agent Workflow

Read this sequence before running the code. The model makes decisions, but the Python program controls what it may do and when the workflow ends.

| Step | What happens | Agent concept shown |
| --- | --- | --- |
| 1. Initialize | The program starts `messages` with the instructions and user task. It starts `working_memory` as an empty evidence list. | Inputs versus instructions; role and goal; two types of memory |
| 2. Decide | The model returns structured JSON that either requests one available tool or provides a final review. | Model tool selection; structured output |
| 3. Validate and run | The program checks that the requested tool is approved and unused, then runs only that reader. | Tool validation; program control |
| 4. Remember | The program adds the approved tool result to `working_memory`. On the next turn, it places that current evidence list into `messages`. | Evidence working memory; conversation memory |
| 5. Repeat or stop | The model receives the updated evidence and decides again. The program permits at most three tool calls, then requires finalization. | Multi-turn loop; stop condition |
| 6. Review | The model returns observations, unknowns, one next step, and a human-review indicator. A person must review the result before taking action. | Evidence boundary; human review |

### Important Notes

- The three approved tools are the maximum tools the workflow may use, not a required checklist. The model may return a final review before using every tool.
- The program prevents unapproved tools and prevents any approved tool from being used more than once.
- `messages` keeps the full conversation history. `working_memory` keeps only approved tool results; the next turn adds that current evidence list to `messages` without duplicating a separate tool-result message.
- Every model decision, including the final review, is returned as structured JSON. The final review contains observations, unknowns, one next step, and a human-review indicator.
- Tool results do not all have the same format: the case brief is plain text, while the manifest and device events are JSON-formatted text.

## Step 1: Setup

This notebook must run from this lab folder so it can find `.env` and the synthetic case packet.

In [ ]:
import csv
import inspect
import json
from collections.abc import Callable
from pathlib import Path
from time import perf_counter

from dotenv import dotenv_values
from openai import OpenAI

LAB_NAME = 'lab0_04_ai_agent'
lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Expected .env in this folder. Copy .env.example to .env first.')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError(f'Could not find {LAB_NAME}/data')

config = dotenv_values(env_path)
model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')
if not model or not ollama_base_url:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from this lab's .env")

client = OpenAI(base_url=ollama_base_url, api_key='ollama')
print('Model:', model)
print('Ollama endpoint:', ollama_base_url)

## Step 2: Define the Approved Forensic Tools

These readers use the `Tool` and `@tool` pattern introduced earlier in this lab. The model receives their descriptions, but it cannot run Python directly. The agent program validates and executes any requested tool.

In [ ]:
def type_name(annotation) -> str:
    return getattr(annotation, '__name__', str(annotation))


class Tool:
    """A reusable wrapper around one approved Python function."""

    def __init__(self, name: str, description: str, function: Callable, arguments: list[str], output: str):
        self.name = name
        self.description = description
        self.function = function
        self.arguments = arguments
        self.output = output

    def to_string(self) -> str:
        arguments_text = ', '.join(self.arguments) if self.arguments else 'none'
        return f'Name: {self.name}\nDescription: {self.description}\nArguments: {arguments_text}\nOutput: {self.output}'

    def __call__(self, *arguments, **keyword_arguments):
        return self.function(*arguments, **keyword_arguments)


def tool(function: Callable) -> Tool:
    """Turn a typed, documented function into a Tool object."""
    signature = inspect.signature(function)
    arguments = [f'{parameter.name}: {type_name(parameter.annotation)}' for parameter in signature.parameters.values()]
    return Tool(function.__name__, inspect.getdoc(function) or 'No description provided.', function, arguments, type_name(signature.return_annotation))


@tool
def read_case_brief() -> str:
    """Read the synthetic case brief."""
    return (data_dir / 'case_brief.md').read_text(encoding='utf-8').strip()


@tool
def read_manifest() -> str:
    """Read the synthetic artifact manifest."""
    manifest = json.loads((data_dir / 'artifact_manifest.json').read_text(encoding='utf-8'))
    return json.dumps(manifest, indent=2)


@tool
def read_device_events() -> str:
    """Read the synthetic device-activity event log."""
    with (data_dir / 'triage_events.csv').open('r', encoding='utf-8', newline='') as handle:
        return json.dumps(list(csv.DictReader(handle)), indent=2)


APPROVED_TOOLS = {tool.name: tool for tool in [read_case_brief, read_manifest, read_device_events]}

for approved_tool in APPROVED_TOOLS.values():
    print(approved_tool.to_string())
    print()

## Step 3: Run the Bounded Forensic Agent

The agent starts with empty working memory. On each turn, the model either requests one available tool or returns its final review. The agent validates the request, runs only approved tools, and saves each result to working memory. At most three tool calls are allowed—one per tool. Once none remain, the model receives one finalization turn and cannot request another tool.

### What is a turn?

A turn is one decision cycle: the program sends the model the current instructions, available tools, and working memory; the model then replies with one tool request or its final review. If it requests a tool, the program validates and runs that tool, adds its result to working memory, and begins the next turn.

Turns let the model reconsider its next step after receiving new evidence. They also keep the workflow inspectable: the program can record each decision, validate every tool request, preserve the evidence the model received, and limit the number of tool calls.

### Read Step 3 in Three Parts

The next code cell has three labeled blocks: **3.1** defines the structured decision format and persistent instructions; **3.2** starts the conversation and evidence memories; and **3.3** runs the bounded decision loop. Run the full cell, then trace one block at a time.

In [ ]:
# Step 3.1: Define the model's decision format and persistent instructions.
# The schema makes model decisions machine-readable before Python reads them.
decision_schema = {
    'type': 'object',
    'properties': {
        'action': {'type': 'string', 'enum': ['tool', 'final']},
        'tool_name': {'type': 'string', 'enum': ['read_case_brief', 'read_manifest', 'read_device_events', '']},
        'observations': {'type': 'array', 'items': {'type': 'string'}},
        'unknowns': {'type': 'array', 'items': {'type': 'string'}},
        'next_step': {'type': 'string'},
        'needs_human_review': {'type': 'boolean'},
    },
    'required': ['action', 'tool_name', 'observations', 'unknowns', 'next_step', 'needs_human_review'],
    'additionalProperties': False,
}

tool_descriptions = '\n\n'.join(tool.to_string() for tool in APPROVED_TOOLS.values())
agent_instructions = f'''
You are the Mobile Device Activity Review Agent.

Role: Produce a cautious first-pass review of a synthetic clinic-phone case.
Goal: Gather only the needed packet information, state supported observations and unknowns, recommend one next human-review step, and stop.

Approved tools:
{tool_descriptions}

Rules:
- Request only one available tool at a time.
- Use tool results as evidence; do not invent facts.
- Do not claim screenshot contents, recipient viewing, or successful delivery unless a tool result directly confirms them.
- After you have enough evidence, return action='final'.
- For action='tool', set tool_name to one available tool and use empty observations, unknowns, and next_step.
- For action='final', set tool_name to an empty string and provide observations, unknowns, one next_step, and needs_human_review=true.
'''.strip()

# Step 3.2: Start the two short-lived memories.
# messages holds the conversation; working_memory holds only approved evidence.
messages = [
    {'role': 'system', 'content': agent_instructions},
    {'role': 'user', 'content': 'Review the synthetic clinic-phone case. Summarize supported device activity, state what remains unknown, and recommend one next human-review step.'},
]

working_memory = []
used_tools = set()
agent_trace = []
final_review = None

# Step 3.3: Run the bounded decision loop.
for turn in range(len(APPROVED_TOOLS) + 1):
    available_tools = [name for name in APPROVED_TOOLS if name not in used_tools]
    instruction = 'Request one available tool or return the final review.' if available_tools else 'No tools remain. Return the final review now.'
    turn_context = {'working_memory': working_memory, 'available_tools': available_tools, 'instruction': instruction}
    messages.append({'role': 'user', 'content': json.dumps(turn_context, indent=2)})

    start = perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        response_format={'type': 'json_schema', 'json_schema': {'name': 'forensic_agent_decision', 'strict': True, 'schema': decision_schema}},
    )
    elapsed = round(perf_counter() - start, 2)
    raw_text = response.choices[0].message.content
    decision = json.loads(raw_text)
    agent_trace.append({'turn': turn + 1, 'seconds': elapsed, 'decision': decision})
    messages.append({'role': 'assistant', 'content': raw_text})

    print(f'\nTurn {turn + 1} decision ({elapsed} seconds):')
    print(json.dumps(decision, indent=2))

    if decision['action'] == 'final':
        final_review = decision
        print('\nStop condition reached: final review returned.')
        break

    tool_name = decision['tool_name']
    if tool_name not in available_tools:
        raise ValueError(f'The model requested an unavailable tool: {tool_name!r}')

    tool_result = APPROVED_TOOLS[tool_name]()
    used_tools.add(tool_name)
    working_memory.append({'tool': tool_name, 'result': tool_result})
    print(f'Approved tool executed: {tool_name}')

if final_review is None:
    raise RuntimeError('The agent did not return a final review before the tool limit.')

## Step 4: Inspect the Bounded Result

The final review is useful only if it follows the required structure and explicitly requires human review. This check verifies those rules, then shows the final review, tool trace, and working memory that supported it.

In [ ]:
required_final_fields = ['observations', 'unknowns', 'next_step', 'needs_human_review']
missing_fields = [field for field in required_final_fields if field not in final_review]
if missing_fields or final_review['action'] != 'final' or final_review['tool_name'] != '' or not final_review['needs_human_review']:
    raise ValueError({'missing_fields': missing_fields, 'action': final_review.get('action'), 'tool_name': final_review.get('tool_name'), 'needs_human_review': final_review.get('needs_human_review')})

print('Final review:')
print(json.dumps(final_review, indent=2))
print('\nTools used:', sorted(used_tools))
print('Working-memory records:', len(working_memory))
print('Agent turns:', len(agent_trace))

## Step 5: Reflection Questions

Replace this text with short answers to the questions below.

1. Pinpoint the code for each agent component: role and goal, approved tools, conversation memory, evidence working memory, tool validation, stop condition, structured output, and human-review boundary. For each, name the variable, function, or code section and state its purpose.
2. Which forensic tool did the model request first, and why was that a reasonable choice?
3. How did the agent program prevent the model from running an unapproved tool?
4. What information entered working memory after each tool result?
5. Which boundary—role, approved tools, memory, stop condition, structured output, or human review—most improved the final review?
6. What conclusion did the workflow deliberately avoid making, and why?

## Submission

Save the notebook with the approved-tool descriptions, printed agent decisions and tool results, final structured review, and your reflection answers.